# MBE Dynamics in a Rollout
Inspect how Matrix-Based Entropy (MBE) evolves as the model generates tokens.
For each hidden layer, we compute MBE on prefixes of increasing length,
revealing how representation diversity builds up during chain-of-thought.

**Key questions:**
1. Does MBE increase monotonically as more tokens are generated?
2. Which layers show the sharpest MBE dynamics?
3. Do correct vs incorrect rollouts differ in their MBE trajectory?

In [1]:
import os
import sys
import re
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

sys.path.insert(0, os.path.abspath("."))
from src.mbe import OnlineMBE, mbe_reverse_gram

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Load Model & Data
Load a Qwen3 model and a few GSM8K examples. Adjust `MODEL_NAME` to point at a checkpoint if available.

In [2]:
# ---- Configuration ----
MODEL_NAME = "Qwen/Qwen3-0.6B"   # swap for a checkpoint path if available
MAX_NEW_TOKENS = 512
N_SAMPLES = 6                     # number of GSM8K problems to rollout
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load model ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map=DEVICE
)
model.eval()
print(f"Model: {MODEL_NAME}  |  Layers: {model.config.num_hidden_layers}  |  Device: {DEVICE}")

# ---- Load GSM8K test set ----
dataset = load_dataset("openai/gsm8k", "main")["test"]
print(f"GSM8K test set: {len(dataset)} examples")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model: Qwen/Qwen3-0.6B  |  Layers: 28  |  Device: cpu


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test set: 1319 examples


## 2. Helper Functions
Generate a rollout and extract hidden states, then compute MBE on prefixes of increasing length using `OnlineMBE`.

In [3]:
def extract_gold_answer(answer_text: str) -> str:
    match = re.search(r"####\s*(.+)", answer_text)
    return match.group(1).strip().replace(",", "") if match else ""

def extract_predicted_answer(text: str) -> str:
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    if match:
        return match.group(1).strip().replace(",", "")
    numbers = re.findall(r"-?[\d,]+\.?\d*", text)
    return numbers[-1].replace(",", "") if numbers else ""

@torch.no_grad()
def generate_rollout(model, tokenizer, question, max_new_tokens=512):
    """Generate a completion and return (completion_text, full_ids, prompt_len)."""
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
    )
    completion_text = tokenizer.decode(output_ids[0, prompt_len:], skip_special_tokens=True)
    return completion_text, output_ids, prompt_len

@torch.no_grad()
def get_hidden_states(model, input_ids):
    """Forward pass returning all hidden states. Returns list of (1, T, D) tensors."""
    outputs = model(input_ids, output_hidden_states=True, use_cache=False)
    return outputs.hidden_states  # tuple: [embedding, layer1, ..., layerN]

def compute_prefix_mbe_online(hidden_states, prompt_len, stride=4):
    """
    Compute MBE at increasing prefix lengths using OnlineMBE.
    
    For each layer, we start from the first completion token and incrementally
    add `stride` tokens at a time, recording MBE after each addition.
    
    Returns:
        positions: list of prefix end positions (relative to completion start)
        mbe_matrix: np.array of shape (n_layers, n_positions) with MBE values
    """
    n_layers = len(hidden_states) - 1  # skip embedding
    seq_len = hidden_states[1].shape[1]
    comp_len = seq_len - prompt_len
    
    if comp_len < stride:
        return [], np.zeros((n_layers, 0))
    
    # Positions where we record MBE (in completion-token space)
    positions = list(range(stride, comp_len + 1, stride))
    if positions and positions[-1] != comp_len:
        positions.append(comp_len)
    
    mbe_matrix = np.zeros((n_layers, len(positions)))
    
    for layer_idx in range(n_layers):
        h = hidden_states[layer_idx + 1][0, prompt_len:, :].float()  # (comp_len, D)
        D = h.shape[1]
        tracker = OnlineMBE(D, device=h.device)
        
        pos_idx = 0
        for t in range(comp_len):
            tracker.update(h[t])
            if pos_idx < len(positions) and (t + 1) == positions[pos_idx]:
                mbe_matrix[layer_idx, pos_idx] = tracker.mbe().item()
                pos_idx += 1
    
    return positions, mbe_matrix

def run_single_example(model, tokenizer, question, gold_answer, max_new_tokens=512, stride=4):
    """Run a full rollout and compute MBE dynamics."""
    completion_text, output_ids, prompt_len = generate_rollout(model, tokenizer, question, max_new_tokens)
    
    predicted = extract_predicted_answer(completion_text)
    try:
        correct = float(predicted) == float(gold_answer)
    except (ValueError, TypeError):
        correct = False
    
    hidden_states = get_hidden_states(model, output_ids)
    positions, mbe_matrix = compute_prefix_mbe_online(hidden_states, prompt_len, stride=stride)
    
    return {
        "question": question[:100],
        "gold": gold_answer,
        "predicted": predicted,
        "correct": correct,
        "comp_len": output_ids.shape[1] - prompt_len,
        "completion_text": completion_text[:200],
        "positions": positions,
        "mbe_matrix": mbe_matrix,  # (n_layers, n_positions)
    }

print("Helpers ready.")

Helpers ready.


## 3. Run Rollouts
Generate completions for a few GSM8K problems and compute per-layer MBE dynamics.

In [ ]:
torch.manual_seed(42)
indices = torch.randperm(len(dataset))[:N_SAMPLES].tolist()

results = []
for i, idx in enumerate(indices):
    example = dataset[idx]
    gold = extract_gold_answer(example["answer"])
    r = run_single_example(model, tokenizer, example["question"], gold, MAX_NEW_TOKENS, stride=4)
    results.append(r)
    tag = "✓" if r["correct"] else "✗"
    print(f"[{i+1}/{N_SAMPLES}] {tag}  pred={r['predicted']:>8s}  gold={r['gold']:>8s}  len={r['comp_len']}")

correct_results = [r for r in results if r["correct"]]
incorrect_results = [r for r in results if not r["correct"]]
print(f"\nCorrect: {len(correct_results)}  |  Incorrect: {len(incorrect_results)}")

[1/6] ✗  pred=          gold=      33  len=512


## 4. Visualize MBE Dynamics

### 4a. Single-rollout heatmap
For each rollout, show a heatmap of MBE (layer × prefix length).

In [ ]:
def plot_mbe_heatmap(result, ax=None):
    """Plot a heatmap of MBE (layer × prefix length) for a single rollout."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 6))
    
    mbe = result["mbe_matrix"]  # (n_layers, n_positions)
    positions = result["positions"]
    n_layers = mbe.shape[0]
    
    if mbe.shape[1] == 0:
        ax.text(0.5, 0.5, "No data (completion too short)", ha="center", va="center")
        return
    
    im = ax.imshow(mbe, aspect="auto", cmap="YlOrRd", interpolation="nearest",
                   origin="lower")
    
    # X-axis: prefix positions (subsample labels to avoid clutter)
    n_xticks = min(15, len(positions))
    xtick_idx = np.linspace(0, len(positions) - 1, n_xticks, dtype=int)
    ax.set_xticks(xtick_idx)
    ax.set_xticklabels([positions[i] for i in xtick_idx], fontsize=8)
    ax.set_xlabel("Prefix length (completion tokens)")
    
    # Y-axis: layers
    ax.set_yticks(range(0, n_layers, max(1, n_layers // 10)))
    ax.set_ylabel("Layer")
    
    tag = "✓ Correct" if result["correct"] else "✗ Incorrect"
    ax.set_title(f"{tag}  |  gold={result['gold']}  pred={result['predicted']}  |  len={result['comp_len']}")
    
    plt.colorbar(im, ax=ax, label="MBE", shrink=0.8)

# Plot heatmaps for all rollouts
n = len(results)
fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), constrained_layout=True)
if n == 1:
    axes = [axes]
for r, ax in zip(results, axes):
    plot_mbe_heatmap(r, ax=ax)
fig.suptitle("MBE Dynamics: Layer × Prefix Length", fontsize=14, y=1.01)
plt.show()

### 4b. Per-layer MBE trajectory (selected layers)
Show MBE vs prefix length for a few representative layers (early, middle, late).

In [ ]:
def plot_layer_trajectories(result, layers=None, ax=None):
    """Plot MBE vs prefix length for selected layers."""
    mbe = result["mbe_matrix"]
    positions = result["positions"]
    n_layers = mbe.shape[0]
    
    if mbe.shape[1] == 0:
        return
    
    if layers is None:
        # Pick ~5 representative layers: early, mid-early, mid, mid-late, late
        layers = sorted(set([
            0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1
        ]))
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    
    cmap = plt.cm.viridis
    for i, layer in enumerate(layers):
        color = cmap(i / max(1, len(layers) - 1))
        ax.plot(positions, mbe[layer], label=f"Layer {layer + 1}", color=color, linewidth=2)
    
    ax.set_xlabel("Prefix length (completion tokens)")
    ax.set_ylabel("MBE")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    tag = "✓" if result["correct"] else "✗"
    ax.set_title(f"{tag}  gold={result['gold']}  pred={result['predicted']}")

# Plot for each rollout
n = len(results)
fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n), constrained_layout=True)
if n == 1:
    axes = [axes]
for r, ax in zip(results, axes):
    plot_layer_trajectories(r, ax=ax)
fig.suptitle("Per-Layer MBE Trajectory", fontsize=14, y=1.01)
plt.show()

### 4c. MBE rate of change (ΔMBE)
Show how fast MBE is growing at each position — the "derivative" of the MBE trajectory.

In [ ]:
def plot_mbe_rate(result, layers=None, ax=None):
    """Plot ΔMBE (rate of change) vs prefix length for selected layers."""
    mbe = result["mbe_matrix"]
    positions = result["positions"]
    n_layers = mbe.shape[0]
    
    if mbe.shape[1] < 2:
        return
    
    if layers is None:
        layers = sorted(set([
            0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1
        ]))
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    
    cmap = plt.cm.viridis
    mid_positions = [(positions[i] + positions[i+1]) / 2 for i in range(len(positions) - 1)]
    
    for i, layer in enumerate(layers):
        delta = np.diff(mbe[layer])
        color = cmap(i / max(1, len(layers) - 1))
        ax.plot(mid_positions, delta, label=f"Layer {layer + 1}", color=color, linewidth=1.5, alpha=0.8)
    
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Prefix length (completion tokens)")
    ax.set_ylabel("ΔMBE (per stride)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    tag = "✓" if result["correct"] else "✗"
    ax.set_title(f"{tag}  ΔMBE rate  |  gold={result['gold']}  pred={result['predicted']}")

# Plot for first 3 rollouts
n_show = min(3, len(results))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4 * n_show), constrained_layout=True)
if n_show == 1:
    axes = [axes]
for r, ax in zip(results[:n_show], axes):
    plot_mbe_rate(r, ax=ax)
fig.suptitle("MBE Rate of Change (ΔMBE per stride)", fontsize=14, y=1.01)
plt.show()

## 5. Correct vs Incorrect: Averaged MBE Dynamics
Average the MBE trajectories across correct and incorrect rollouts to see systematic differences.
We interpolate all rollouts to a common normalized position axis (0–100% of completion).

In [ ]:
def interpolate_mbe(result, n_points=50):
    """Interpolate MBE matrix to a common normalized axis [0, 1]."""
    mbe = result["mbe_matrix"]
    positions = np.array(result["positions"], dtype=float)
    if mbe.shape[1] < 2:
        return None
    
    norm_pos = positions / positions[-1]  # normalize to [0, 1]
    common_axis = np.linspace(0, 1, n_points)
    
    n_layers = mbe.shape[0]
    interp = np.zeros((n_layers, n_points))
    for layer in range(n_layers):
        interp[layer] = np.interp(common_axis, norm_pos, mbe[layer])
    return interp

N_POINTS = 50
common_axis = np.linspace(0, 100, N_POINTS)  # 0–100% of completion

correct_interp = [interpolate_mbe(r, N_POINTS) for r in correct_results]
correct_interp = [x for x in correct_interp if x is not None]

incorrect_interp = [interpolate_mbe(r, N_POINTS) for r in incorrect_results]
incorrect_interp = [x for x in incorrect_interp if x is not None]

has_correct = len(correct_interp) > 0
has_incorrect = len(incorrect_interp) > 0

if has_correct:
    correct_mean = np.mean(correct_interp, axis=0)   # (n_layers, N_POINTS)
if has_incorrect:
    incorrect_mean = np.mean(incorrect_interp, axis=0)

n_layers = results[0]["mbe_matrix"].shape[0]
print(f"Correct rollouts: {len(correct_interp)}  |  Incorrect rollouts: {len(incorrect_interp)}")
print(f"Layers: {n_layers}  |  Interpolation points: {N_POINTS}")

In [ ]:
# --- 5a. Averaged MBE heatmaps: Correct vs Incorrect side by side ---
fig, axes = plt.subplots(1, 2 if (has_correct and has_incorrect) else 1,
                         figsize=(14 if (has_correct and has_incorrect) else 8, 6),
                         constrained_layout=True)
if not isinstance(axes, np.ndarray):
    axes = [axes]

plot_idx = 0
vmin = min(correct_mean.min() if has_correct else 999, incorrect_mean.min() if has_incorrect else 999)
vmax = max(correct_mean.max() if has_correct else 0, incorrect_mean.max() if has_incorrect else 0)

if has_correct:
    im = axes[plot_idx].imshow(correct_mean, aspect="auto", cmap="YlOrRd",
                                origin="lower", vmin=vmin, vmax=vmax)
    axes[plot_idx].set_title(f"Correct (n={len(correct_interp)})")
    axes[plot_idx].set_xlabel("Completion progress (%)")
    axes[plot_idx].set_ylabel("Layer")
    n_xticks = 6
    xtick_idx = np.linspace(0, N_POINTS - 1, n_xticks, dtype=int)
    axes[plot_idx].set_xticks(xtick_idx)
    axes[plot_idx].set_xticklabels([f"{common_axis[i]:.0f}" for i in xtick_idx])
    plot_idx += 1

if has_incorrect:
    im = axes[plot_idx].imshow(incorrect_mean, aspect="auto", cmap="YlOrRd",
                                origin="lower", vmin=vmin, vmax=vmax)
    axes[plot_idx].set_title(f"Incorrect (n={len(incorrect_interp)})")
    axes[plot_idx].set_xlabel("Completion progress (%)")
    axes[plot_idx].set_ylabel("Layer")
    axes[plot_idx].set_xticks(xtick_idx)
    axes[plot_idx].set_xticklabels([f"{common_axis[i]:.0f}" for i in xtick_idx])

fig.colorbar(im, ax=axes, label="MBE", shrink=0.8)
fig.suptitle("Averaged MBE Dynamics: Correct vs Incorrect", fontsize=14)
plt.show()

In [ ]:
# --- 5b. Per-layer averaged trajectories: Correct vs Incorrect ---
layers_to_show = sorted(set([
    0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1
]))

fig, axes = plt.subplots(1, len(layers_to_show), figsize=(5 * len(layers_to_show), 4),
                         constrained_layout=True, sharey=True)

for i, layer in enumerate(layers_to_show):
    ax = axes[i]
    if has_correct:
        ax.plot(common_axis, correct_mean[layer], color="#27ae60", linewidth=2, label="Correct")
        if len(correct_interp) > 1:
            correct_std = np.std([x[layer] for x in correct_interp], axis=0)
            ax.fill_between(common_axis, correct_mean[layer] - correct_std,
                           correct_mean[layer] + correct_std, color="#27ae60", alpha=0.15)
    if has_incorrect:
        ax.plot(common_axis, incorrect_mean[layer], color="#e74c3c", linewidth=2, label="Incorrect")
        if len(incorrect_interp) > 1:
            incorrect_std = np.std([x[layer] for x in incorrect_interp], axis=0)
            ax.fill_between(common_axis, incorrect_mean[layer] - incorrect_std,
                           incorrect_mean[layer] + incorrect_std, color="#e74c3c", alpha=0.15)
    
    ax.set_title(f"Layer {layer + 1}", fontsize=12)
    ax.set_xlabel("Completion %")
    if i == 0:
        ax.set_ylabel("MBE")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("MBE Trajectory by Layer: Correct vs Incorrect", fontsize=14, y=1.02)
plt.show()

### 5c. Delta heatmap (Correct − Incorrect)
Where in the layer × position space does MBE differ most between correct and incorrect rollouts?

In [ ]:
# --- 5c. Delta heatmap: Correct − Incorrect ---
if has_correct and has_incorrect:
    delta = correct_mean - incorrect_mean
    abs_max = max(abs(delta.min()), abs(delta.max()))
    
    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    im = ax.imshow(delta, aspect="auto", cmap="RdBu_r", origin="lower",
                   vmin=-abs_max, vmax=abs_max)
    
    n_xticks = 8
    xtick_idx = np.linspace(0, N_POINTS - 1, n_xticks, dtype=int)
    ax.set_xticks(xtick_idx)
    ax.set_xticklabels([f"{common_axis[i]:.0f}%" for i in xtick_idx])
    ax.set_xlabel("Completion progress (%)")
    ax.set_ylabel("Layer")
    ax.set_title("ΔMBE (Correct − Incorrect): Red = correct has higher MBE", fontsize=13)
    plt.colorbar(im, ax=ax, label="ΔMBE", shrink=0.8)
    plt.show()
else:
    print("Need both correct and incorrect rollouts for delta heatmap.")

## 6. Summary: MBE Growth Profile
For each layer, compute the total MBE growth (final − initial) and the position where MBE crosses 50% of its final value ("half-life"). This reveals which layers build diversity early vs late.

In [ ]:
# --- 6. MBE growth profile per layer ---
def compute_growth_profile(mbe_matrix, positions):
    """For each layer, compute total growth and half-life position."""
    n_layers = mbe_matrix.shape[0]
    growth = []
    half_life = []
    
    for layer in range(n_layers):
        trace = mbe_matrix[layer]
        if len(trace) < 2:
            growth.append(0.0)
            half_life.append(0.0)
            continue
        
        total = trace[-1] - trace[0]
        growth.append(total)
        
        # Find position where MBE crosses 50% of final value
        mid = trace[0] + total * 0.5
        crossed = np.where(trace >= mid)[0]
        if len(crossed) > 0:
            half_life.append(positions[crossed[0]] / positions[-1] * 100)  # as % of completion
        else:
            half_life.append(100.0)
    
    return np.array(growth), np.array(half_life)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

layers = np.arange(1, n_layers + 1)

if has_correct:
    g_c, hl_c = zip(*[compute_growth_profile(r["mbe_matrix"], r["positions"]) for r in correct_results if len(r["positions"]) > 1])
    g_c_mean, hl_c_mean = np.mean(g_c, axis=0), np.mean(hl_c, axis=0)
    ax1.plot(layers, g_c_mean, color="#27ae60", linewidth=2, label="Correct", marker="o", markersize=3)
    ax2.plot(layers, hl_c_mean, color="#27ae60", linewidth=2, label="Correct", marker="o", markersize=3)

if has_incorrect:
    g_i, hl_i = zip(*[compute_growth_profile(r["mbe_matrix"], r["positions"]) for r in incorrect_results if len(r["positions"]) > 1])
    g_i_mean, hl_i_mean = np.mean(g_i, axis=0), np.mean(hl_i, axis=0)
    ax1.plot(layers, g_i_mean, color="#e74c3c", linewidth=2, label="Incorrect", marker="o", markersize=3)
    ax2.plot(layers, hl_i_mean, color="#e74c3c", linewidth=2, label="Incorrect", marker="o", markersize=3)

ax1.set_xlabel("Layer")
ax1.set_ylabel("MBE growth (final − initial)")
ax1.set_title("Total MBE Growth per Layer")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Layer")
ax2.set_ylabel("Half-life position (% of completion)")
ax2.set_title("MBE Half-Life: Where Does Diversity Build Up?")
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle("MBE Growth Profile by Layer", fontsize=14, y=1.02)
plt.show()